# Week 6: Unsupervised Learning & Dimensionality Reduction
## Notebook 07: Gaussian Mixture Models (GMM) & the EM Algorithm from Scratch

### 1. Conceptual Intuition & Real-World Use Case
K-Means performs "Hard Clustering": every data point belongs 100% to exactly one centroid.
However, real-world data is ambiguous:
- A customer might have a 60% probability of being a "Casual Shopper" and a 40% probability of being a "Bargain Hunter".
- Clusters often have non-spherical elliptical shapes with different variances along different axes.

**Gaussian Mixture Models (GMM)** are probabilistic models assuming data is generated from a mixture of $K$ Gaussian distributions.
They perform **Soft Clustering** using the **Expectation-Maximization (EM)** algorithm.

---

### 2. Mathematical Foundations: Expectation-Maximization (EM)
Probability density of sample $x$:
$$p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x \mid \mu_k, \Sigma_k)$$
Where $\pi_k$ is the mixture weight ($sum \pi_k = 1$).

#### 1. E-Step (Expectation - Responsibilities):
Compute posterior probability $\gamma_{ik}$ that Gaussian component $k$ generated point $x_i$:
$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}$$

#### 2. M-Step (Maximization - Update Parameters):
$$\mu_k^{new} = \frac{\sum_i \gamma_{ik} x_i}{N_k}, \quad \Sigma_k^{new} = \frac{\sum_i \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T}{N_k}, \quad \pi_k^{new} = \frac{N_k}{N}$$
where $N_k = \sum_{i=1}^N \gamma_{ik}$ is the effective number of points assigned to cluster $k$.

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal

np.random.seed(42)

# Step 1: Generate 1D Data from 2 Overlapping Gaussians
# Gaussian 1: mu = 2.0, sigma = 0.8
# Gaussian 2: mu = 7.0, sigma = 1.5
X1 = np.random.normal(2.0, 0.8, size=150)
X2 = np.random.normal(7.0, 1.5, size=250)
X = np.concatenate([X1, X2])
N = len(X)

print(f"Total points generated: {N} (Mixture of 2 Gaussians)")

# Step 2: Implement 1D Expectation-Maximization (EM) from Scratch
# Initialize parameters
K = 2
pi = np.array([0.5, 0.5])
mu = np.array([1.0, 5.0])
sigma = np.array([1.0, 1.0])

def gaussian_pdf(x, m, s):
    return (1.0 / (np.sqrt(2 * np.pi) * s)) * np.exp(-0.5 * ((x - m) / s) ** 2)

epochs = 30
for epoch in range(epochs):
    # E-STEP: Compute responsibilities gamma (N, K)
    likelihood = np.zeros((N, K))
    for k in range(K):
        likelihood[:, k] = pi[k] * gaussian_pdf(X, mu[k], sigma[k])
    
    # Normalize across components
    gamma = likelihood / np.sum(likelihood, axis=1, keepdims=True)
    
    # M-STEP: Update mu, sigma, pi
    N_k = np.sum(gamma, axis=0)
    
    for k in range(K):
        mu[k] = np.sum(gamma[:, k] * X) / N_k[k]
        sigma[k] = np.sqrt(np.sum(gamma[:, k] * (X - mu[k]) ** 2) / N_k[k])
        pi[k] = N_k[k] / N

print("--- EM Learned Parameters vs Ground Truth ---")
print(f"Component 0: Learned mu = {mu[0]:.3f} (True: 2.00) | sigma = {sigma[0]:.3f} (True: 0.80) | pi = {pi[0]:.3f}")
print(f"Component 1: Learned mu = {mu[1]:.3f} (True: 7.00) | sigma = {sigma[1]:.3f} (True: 1.50) | pi = {pi[1]:.3f}")

# Step 3: Soft Clustering Probabilities for Boundary Points
test_points = np.array([2.0, 4.5, 7.0])
for pt in test_points:
    p0 = pi[0] * gaussian_pdf(pt, mu[0], sigma[0])
    p1 = pi[1] * gaussian_pdf(pt, mu[1], sigma[1])
    prob_c0 = p0 / (p0 + p1)
    prob_c1 = p1 / (p0 + p1)
    print(f"\nPoint {pt}: P(Cluster 0) = {prob_c0 * 100:.1f}%, P(Cluster 1) = {prob_c1 * 100:.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **E-Step:** Computes soft membership weights $\gamma$ using current Gaussian density estimates.
2. **M-Step:** Recomputes means and variances weighted by the responsibilities $\gamma_{ik}$.
3. **Soft Assignment:** Notice point $4.5$ (in the overlap boundary) receives nuanced soft probabilities ($31\% / 69\%$) instead of an arbitrary binary cutoff.

---

### 4. Industry Applications & Trade-offs
- **Density Estimation & Anomaly Detection:** Computing the log-likelihood of new transactions; points with very low $p(x)$ are flagged as novel anomalies.
- **Trade-off:** EM is sensitive to initialization and can get trapped in local optima; initializing with K-Means centroids is standard practice.

---

### 5. Top Interview Questions & Answers
**Q1: How does K-Means relate mathematically to Gaussian Mixture Models?**
> **Answer:** K-Means is a special limiting case of GMM where: (1) All clusters have identical, fixed spherical covariance $\Sigma_k = \sigma^2 I$, (2) Equal mixture weights $\pi_k = 1/K$, and (3) Variance $\sigma^2 \to 0$. As $\sigma^2 \to 0$, soft responsibilities converge to hard binary 0/1 assignments.

**Q2: What is the purpose of BIC (Bayesian Information Criterion) in GMM?**
> **Answer:** BIC balances model fit (log-likelihood) against model complexity (number of parameters): $\text{BIC} = k \ln(N) - 2 \ln(\hat{L})$. It prevents overfitting by heavily penalizing larger numbers of Gaussian components, enabling automated selection of the optimal $K$.